# 13주차 복습과제 — Dropout 구현

> **참고 논문:** Srivastava et al., *Dropout: A Simple Way to Prevent Neural Networks from Overfitting*, JMLR 2014  
> **데이터셋:** Fashion-MNIST  

---

## 🎯 과제 목표

이번 과제는 딥러닝 정규화 기법인 **Dropout**을 직접 구현해봄으로써 논문에서 제안한 방법을 실습합니다.  
Dropout은 훈련 중 각 뉴런을 확률적으로 제거하여 과적합(Overfitting)을 방지하고, 수많은 서브 네트워크의 앙상블 효과를 내는 기법입니다.

---

## ✏️ Q1. 다음 빈칸을 채우세요.

Dropout은 훈련 중 각 뉴런을 일정 ( ① )로 무작위로 제거하여 매 배치마다 서로 다른 ( ② )를 학습시키는 효과를 낸다. 이는 ( ③ ) 효과를 단일 모델로 근사하며, 뉴런 간 ( ④ )을 방지하여 과적합을 줄인다.

> 보기: `확률` `서브 네트워크` `앙상블` `공동 적응(co-adaptation)`

**답:**

```
① 확률
② 서브 네트워크
③ 앙상블
④ 공동 적응(co-adpatation)
```

## ✏️ Q2. 다음 빈칸을 채우세요.

훈련 시에는 Dropout을 ( ① )하고, 테스트 시에는 Dropout을 ( ② )하는 대신 모든 가중치에 ( ③ )를 곱하여 스케일을 보정한다. 이를 ( ④ )이라고 한다.

> 보기: `적용` `적용하지 않` `(1-p)` `Weight Scaling`

**답:**

```
① 적용
② 적용하지 않
③ (1-p)
④ Weight Scaling
```

---

## 💻 코드 구현 파트

아래 Q3 ~ Q7은 코드의 빈칸(`______`)을 채우는 문제입니다.  
주석을 잘 읽고 올바른 코드를 작성하세요.

> **실행 환경:** Google Colab 또는 로컬 Python 환경 (TensorFlow 2.x)

---

## 💻 Q3. 라이브러리 불러오기 및 데이터 준비

필요한 라이브러리를 import하고 Fashion-MNIST 데이터를 불러온 후 정규화하세요.  
픽셀값의 범위는 0 ~ 255이므로, **255.0으로 나누어** 0 ~ 1 사이로 정규화합니다.

In [1]:
import tensorflow as tf
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Flatten
from tensorflow.keras.callbacks import EarlyStopping

# Fashion-MNIST 데이터셋 불러오기
(train_images, train_labels), (test_images, test_labels) = fashion_mnist.load_data()

# [Q3] 픽셀값을 0~1 사이로 정규화하세요 (255.0으로 나누기)
train_images = train_images / 255.0
test_images  = test_images / 255.0

print("훈련 데이터 shape:", train_images.shape)
print("테스트 데이터 shape:", test_images.shape)
print("픽셀값 범위:", train_images.min(), "~", train_images.max())

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
훈련 데이터 shape: (60000, 28, 28)
테스트 데이터 shape: (10000, 28, 28)
픽셀값 범위: 0.0 ~ 1.0


---

## 💻 Q4. Dropout이 적용된 모델 정의

아래 모델 구조에 맞게 빈칸을 채우세요.

| 레이어 | 설명 |
|--------|------|
| `Flatten` | 28×28 이미지를 1차원(784)으로 펼치기 |
| `Dense(128, relu)` | 은닉층 1 |
| **`Dropout(0.2)`** | 20% 뉴런 제거 (입력층에 가까울수록 낮은 비율 권장) |
| `Dense(64, relu)` | 은닉층 2 |
| **`Dropout(0.3)`** | 30% 뉴런 제거 |
| `Dense(10, softmax)` | 출력층 (10개 클래스) |

In [2]:
# [Q4] 아래 빈칸을 채워 Dropout이 포함된 모델을 완성하세요
model = Sequential([
    Flatten(input_shape=(28, 28)),
    Dense(128, activation='relu'),
    Dropout(0.2),          # 빈칸 1: 첫 번째 Dropout 비율 (0.2)
    Dense(64, activation='relu'),
    Dropout(0.3),           # 빈칸 2~3: 두 번째 Dropout 레이어와 비율 (0.3)
    Dense(10, activation='softmax')
])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       100,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 109,386 (427.29 KB)

 Trainable params: 109,386 (427.29 KB)

 Non-trainable params: 0 (0.00 B)

---

## 💻 Q5. 모델 컴파일

모델을 학습하기 위해 optimizer, loss, metrics를 설정하세요.

- **optimizer:** `'adam'`
- **loss:** `'sparse_categorical_crossentropy'` (레이블이 정수일 때 사용)
- **metrics:** `['accuracy']`

In [3]:
# [Q5] 빈칸을 채워 모델을 컴파일하세요
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
print("모델 컴파일 완료!")

모델 컴파일 완료!


---

## 💻 Q6. 모델 학습 (Early Stopping 포함)

**Early Stopping**은 검증 손실(`val_loss`)이 일정 횟수(`patience`) 동안 개선되지 않으면 학습을 조기 종료하는 기법입니다.  
Dropout과 함께 사용하면 과적합을 더욱 효과적으로 방지할 수 있습니다.

In [4]:
# Early Stopping 설정: val_loss 기준, patience=5, 최적 가중치 복원
early_stopping = EarlyStopping(
    monitor='val_loss',           # 빈칸 4: 모니터링할 지표 ('val_loss')
    patience=5,
    restore_best_weights=True
)

# [Q6] 빈칸을 채워 모델을 학습시키세요
history = model.fit(
    train_images, train_labels,
    epochs=20,
    validation_split=0.2,     # 훈련 데이터의 20%를 검증에 사용
    callbacks=[early_stopping]        # 빈칸 5: early_stopping 콜백 전달
)

Epoch 1/20
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.7788 - loss: 0.6205 - val_accuracy: 0.8367 - val_loss: 0.4411
Epoch 2/20
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.8381 - loss: 0.4524 - val_accuracy: 0.8607 - val_loss: 0.3833
Epoch 3/20
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.8507 - loss: 0.4151 - val_accuracy: 0.8635 - val_loss: 0.3751
Epoch 4/20
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.8592 - loss: 0.3890 - val_accuracy: 0.8678 - val_loss: 0.3563
Epoch 5/20
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.8655 - loss: 0.3703 - val_accuracy: 0.8690 - val_loss: 0.3718
Epoch 6/20
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.8709 - loss: 0.3569 - val_accuracy: 0.8686 - val_loss: 0.3715
Epoch 7/20
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - accuracy: 0.8731 - loss: 0.3477 - val_accuracy: 0.8757 - val_loss: 0.3389
Epoch 8/20
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 8s 6ms/step - accuracy: 0.8762 - loss: 0.3372 - 

---

## 💻 Q7. 모델 평가 및 결과 해석

테스트 데이터셋으로 모델 성능을 평가하고, 아래 결과 해석 질문에 답하세요.

In [5]:
# 테스트 데이터셋으로 모델 평가
test_loss, test_acc = model.evaluate(test_images, test_labels, verbose=2)
print(f"\n테스트 정확도: {test_acc:.4f}")
print(f"테스트 손실:   {test_loss:.4f}")

313/313 - 1s - 2ms/step - accuracy: 0.8823 - loss: 0.3434

테스트 정확도: 0.8823
테스트 손실:   0.3434


**[결과 해석]** 위 결과를 보고 아래 질문에 간단히 답하세요.

1. 테스트 정확도가 약 87~88% 정도 나왔나요? 만약 훈련 정확도보다 테스트 정확도가 많이 낮다면 무엇을 의심할 수 있나요?

```
정확도: 88%

만약 훈련 정확도보다 테스트 정확도가 많이 낮다면 과적합(Overfitting)을 의심할 수 있습니다.
즉, 모델이 훈련 데이터에만 지나치게 맞춰져 새로운 데이터에서는 성능이 떨어지는 상황입니다.
```

2. Dropout 비율을 0.5로 높이면 어떤 효과가 생길 것 같나요?

```
Dropout 비율을 0.5로 높이면 더 많은 뉴런이 제거되어 과적합 방지 효과는 커질 수 있습니다.

하지만 너무 높은 비율은 중요한 정보까지 잃게 만들어 학습이 어려워지고,
훈련 정확도와 테스트 정확도가 모두 낮아질 가능성도 있습니다.
```

---

